В первом таске нам нужно сделать dataset по активности молекул COX-2. 

In [36]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdmolops

In [37]:
df = pd.read_csv('input_file.csv', sep=';')
df = df[['Smiles', 'Standard Relation', 'Standard Units', 'Standard Value']]
df

,Smiles,Standard Relation,Standard Units,Standard Value
0,CC1(C)OC(=O)C(OC2CCCCC2)=C1c1ccc(S(C)(=O)=O)cc1,'=',nM,40.0
1,CCc1ccc(-c2ncc(Cl)cc2-c2ccc(S(C)(=O)=O)cc2)cn1,'=',nM,1700.0
2,CCCCOC(=O)Cc1c(C)n(C(=O)c2ccc(Cl)cc2)c2ccc(OC)...,'=',nM,50.0
3,COc1ccc2c(c1)c(CC(=O)O)c(C)n2C(=O)c1ccc(Cl)cc1,'<',nM,200.0
4,CS(=O)(=O)c1ccc(-c2csc(CC(=O)O)c2-c2ccc(F)cc2)cc1,'>',nM,10000.0
...,...,...,...,...
7974,Cc1ccc(-c2cc(C(F)(F)F)nn2-c2ccc(S(N)(=O)=O)cc2...,'=',nM,40.0
7975,CC(C)Cc1ccc(C(C)C(=O)O)cc1,'=',nM,370000.0
7976,O=C(O)Cc1ccccc1Nc1c(Cl)cccc1Cl,'=',nM,1100.0
7977,Cc1c(Cl)cccc1Nc1ccccc1C(=O)O,'=',nM,880.0


Удаляем NaN строки в Standard Value и Standard Units

In [38]:
df = df.dropna(subset=['Standard Value'])
df = df.dropna(subset=['Standard Units'])
df

,Smiles,Standard Relation,Standard Units,Standard Value
0,CC1(C)OC(=O)C(OC2CCCCC2)=C1c1ccc(S(C)(=O)=O)cc1,'=',nM,40.0
1,CCc1ccc(-c2ncc(Cl)cc2-c2ccc(S(C)(=O)=O)cc2)cn1,'=',nM,1700.0
2,CCCCOC(=O)Cc1c(C)n(C(=O)c2ccc(Cl)cc2)c2ccc(OC)...,'=',nM,50.0
3,COc1ccc2c(c1)c(CC(=O)O)c(C)n2C(=O)c1ccc(Cl)cc1,'<',nM,200.0
4,CS(=O)(=O)c1ccc(-c2csc(CC(=O)O)c2-c2ccc(F)cc2)cc1,'>',nM,10000.0
...,...,...,...,...
7974,Cc1ccc(-c2cc(C(F)(F)F)nn2-c2ccc(S(N)(=O)=O)cc2...,'=',nM,40.0
7975,CC(C)Cc1ccc(C(C)C(=O)O)cc1,'=',nM,370000.0
7976,O=C(O)Cc1ccccc1Nc1c(Cl)cccc1Cl,'=',nM,1100.0
7977,Cc1c(Cl)cccc1Nc1ccccc1C(=O)O,'=',nM,880.0


Проверим какие значения есть в Standard Relation и Standard Units.

In [39]:
print ('Standard Units:', df['Standard Units'].unique().tolist())
print ('Standard Relation', df['Standard Relation'].unique().tolist())

Standard Units: ['nM', 'ug.mL-1', 'ug', '%']
Standard Relation ["'='", "'<'", "'>'", "'>='"]


In [40]:
print ('nM:', (df['Standard Units'] == 'nM').sum())
print ('ug.mL-1:', (df['Standard Units'] == 'ug.mL-1').sum())
print ('ug:', (df['Standard Units'] == 'ug').sum())
print ('%:', (df['Standard Units'] == '%').sum())
print()
print ("'=':", (df['Standard Relation'] == "'='").sum())
print ("'<':", (df['Standard Relation'] == "'<'").sum())
print ("'>':", (df['Standard Relation'] == "'>'").sum())
print ("'>=':", (df['Standard Relation'] == "'>='").sum())

nM: 6951
ug.mL-1: 25
ug: 1
%: 1

'=': 5948
'<': 56
'>': 973
'>=': 1


По поводу Standard Units:
- нам даны четыре единицы измерения, но подавляющее большинство это nM, поэтому мы просто отбросим остальные 27 строк.
 
По поводу Standard Relation:
- значения '<', '>' и '>=' не дают нам четкого ответа, поэтому уберем их и оставим только '='

In [41]:
df_drop = df.copy()

df_drop = df_drop[df_drop['Standard Units'] == 'nM'].reset_index(drop=True)
df_drop = df_drop[df_drop['Standard Relation'] == "'='"].reset_index(drop=True)
df_drop

,Smiles,Standard Relation,Standard Units,Standard Value
0,CC1(C)OC(=O)C(OC2CCCCC2)=C1c1ccc(S(C)(=O)=O)cc1,'=',nM,40.0
1,CCc1ccc(-c2ncc(Cl)cc2-c2ccc(S(C)(=O)=O)cc2)cn1,'=',nM,1700.0
2,CCCCOC(=O)Cc1c(C)n(C(=O)c2ccc(Cl)cc2)c2ccc(OC)...,'=',nM,50.0
3,CC(C)CCCC(=O)c1cc(C(C)(C)C)c2c(c1)C(C)(C)CO2,'=',nM,4500.0
4,COc1ccc2c(c1)c(CC(=O)NCCOC(=O)/C=C/c1ccc(N(C)C...,'=',nM,190.0
...,...,...,...,...
5922,Cc1ccc(-c2cc(C(F)(F)F)nn2-c2ccc(S(N)(=O)=O)cc2...,'=',nM,40.0
5923,CC(C)Cc1ccc(C(C)C(=O)O)cc1,'=',nM,370000.0
5924,O=C(O)Cc1ccccc1Nc1c(Cl)cccc1Cl,'=',nM,1100.0
5925,Cc1c(Cl)cccc1Nc1ccccc1C(=O)O,'=',nM,880.0


Удаляем дубликаты проверяя по столбцу Smiles

In [42]:
df_drop = df_drop.drop_duplicates(subset=['Smiles']).reset_index(drop=True)
df_drop

,Smiles,Standard Relation,Standard Units,Standard Value
0,CC1(C)OC(=O)C(OC2CCCCC2)=C1c1ccc(S(C)(=O)=O)cc1,'=',nM,40.0
1,CCc1ccc(-c2ncc(Cl)cc2-c2ccc(S(C)(=O)=O)cc2)cn1,'=',nM,1700.0
2,CCCCOC(=O)Cc1c(C)n(C(=O)c2ccc(Cl)cc2)c2ccc(OC)...,'=',nM,50.0
3,CC(C)CCCC(=O)c1cc(C(C)(C)C)c2c(c1)C(C)(C)CO2,'=',nM,4500.0
4,COc1ccc2c(c1)c(CC(=O)NCCOC(=O)/C=C/c1ccc(N(C)C...,'=',nM,190.0
...,...,...,...,...
4200,O=C(O)/C=C/c1ccc(O)c(O)c1,'=',nM,129700.0
4201,COc1ccc2c(c1)c(CC(=O)OCC(=O)O)c(C)n2C(=O)c1ccc...,'=',nM,10.0
4202,COc1ccc(/C=C/c2c(CC=C(C)C)c(O)cc(O)c2CC=C(C)C)...,'=',nM,14700.0
4203,Nc1ccc(O)c(C(=O)O)c1,'=',nM,7530.0


Проверка валидности SMILES 

In [44]:
def valid_smiles(smiles):
    try:
        # Попытка создать молекулу из SMILES 
        mol = Chem.MolFromSmiles(smiles, sanitize=False)
        if mol is None:
            return False
        rdmolops.SanitizeMol(mol)
        return True
    except Exception:
        return False

mask = df_drop['Smiles'].apply(valid_smiles)
df_valid = df_drop[mask].reset_index(drop=True)
df_valid

,Smiles,Standard Relation,Standard Units,Standard Value
0,CC1(C)OC(=O)C(OC2CCCCC2)=C1c1ccc(S(C)(=O)=O)cc1,'=',nM,40.0
1,CCc1ccc(-c2ncc(Cl)cc2-c2ccc(S(C)(=O)=O)cc2)cn1,'=',nM,1700.0
2,CCCCOC(=O)Cc1c(C)n(C(=O)c2ccc(Cl)cc2)c2ccc(OC)...,'=',nM,50.0
3,CC(C)CCCC(=O)c1cc(C(C)(C)C)c2c(c1)C(C)(C)CO2,'=',nM,4500.0
4,COc1ccc2c(c1)c(CC(=O)NCCOC(=O)/C=C/c1ccc(N(C)C...,'=',nM,190.0
...,...,...,...,...
4199,O=C(O)/C=C/c1ccc(O)c(O)c1,'=',nM,129700.0
4200,COc1ccc2c(c1)c(CC(=O)OCC(=O)O)c(C)n2C(=O)c1ccc...,'=',nM,10.0
4201,COc1ccc(/C=C/c2c(CC=C(C)C)c(O)cc(O)c2CC=C(C)C)...,'=',nM,14700.0
4202,Nc1ccc(O)c(C(=O)O)c1,'=',nM,7530.0
